# Crawling Berita Detik.com
* Berita Sport : 100 
* Berita Finance : 100

In [13]:
%pip install trafilatura requests beautifulsoup4 pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


## Import Library

In [14]:
import requests
import time
import re
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse, parse_qs, urlencode
import trafilatura

## Konfigurasi

In [15]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7"
}

TARGET_PER_CATEGORY = 100
REQUEST_TIMEOUT = 20
DELAY = 1
MAX_PAGES = 30

## Fungsi Validasi Url

In [16]:
def normalize_url(url):
    """
    Membersihkan URL agar URL yang sama tidak dianggap berbeda.
    Query string (mis. ?page=2) TETAP dipertahankan karena
    dipakai untuk pagination -- membuangnya membuat semua
    halaman index dianggap sama.
    """
    parsed = urlparse(url)

    clean = urlunparse((
        parsed.scheme,
        parsed.netloc.lower(),
        parsed.path.rstrip("/"),
        "",
        parsed.query,
        ""
    ))

    return clean

In [17]:
def is_valid_article_url(url, allowed_domain):
    """
    Memastikan URL benar-benar berasal dari domain kategori
    dan merupakan halaman artikel, bukan halaman indeks/kategori.
    """

    try:
        parsed = urlparse(url)

        domain = parsed.netloc.lower()
        path = parsed.path.lower()

        # Harus HTTPS/HTTP
        if parsed.scheme not in ("http", "https"):
            return False

        # Harus tepat pada domain yang diizinkan
        if domain != allowed_domain:
            return False

        # Jangan ambil halaman indeks
        if "/indeks" in path:
            return False

        # Jangan ambil halaman kategori/tag/search
        forbidden = [
            "/search",
            "/tag/",
            "/foto/",
            "/video/",
            "/live/",
            "/infografis/",
            "/detiktv/"
        ]

        if any(x in path for x in forbidden):
            return False

        # URL harus memiliki path
        if not path or path == "/":
            return False

        # WAJIB: hanya terima halaman artikel asli detik.com,
        # yang selalu punya pola "/d-<angka>/" di URL-nya.
        # Tanpa ini, link kategori seperti /sepakbola, /moto-gp,
        # /f1, dsb ikut lolos dan bikin scrape_article gagal terus.
        if not re.search(r"/d-\d+", path):
            return False

        return True

    except Exception:
        return False

## Fungsi mengambil halaman index

In [18]:
def get_links_from_index(index_url, allowed_domain):
    """
    Mengambil link artikel dari satu halaman indeks.
    """

    links = []

    try:
        response = requests.get(
            index_url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT
        )

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for a in soup.find_all("a", href=True):

            href = a.get("href")

            if not href:
                continue

            # Ubah URL relatif menjadi absolut
            url = urljoin(index_url, href)

            # Normalisasi
            url = normalize_url(url)

            # Validasi
            if not is_valid_article_url(url, allowed_domain):
                continue

            # Hindari duplikat
            if url not in links:
                links.append(url)

        return links

    except requests.RequestException as e:
        print(f"❌ Gagal mengambil index:")
        print(index_url)
        print(f"Error: {e}")
        return []

    except Exception as e:
        print(f"❌ Error parsing index:")
        print(index_url)
        print(f"Error: {e}")
        return []

## Fungsi pagination

In [ ]:
def get_next_page(index_url, allowed_domain):
    """
    Mencari link halaman indeks berikutnya.
    Fungsi ini mengambil sendiri HTML dari index_url karena
    crawl_category tidak lagi meneruskan objek `soup`.
    """

    try:
        response = requests.get(
            index_url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
    except Exception as e:
        print(f"⚠ Gagal mengambil halaman untuk cek pagination: {e}")
        return None

    candidates = []

    for a in soup.find_all("a", href=True):

        href = a.get("href")
        text = a.get_text(" ", strip=True).lower()

        if not href:
            continue

        url = normalize_url(urljoin(index_url, href))

        parsed = urlparse(url)
        domain = parsed.netloc.lower()
        path = parsed.path.lower()

        # Harus domain yang sama
        if domain != allowed_domain:
            continue

        # Harus halaman indeks
        if "/indeks" not in path:
            continue

        # Prioritaskan teks "next"
        if text in ["next", "berikutnya", "selanjutnya", ">"]:
            candidates.append(url)

    if candidates:
        return candidates[0]

    parsed_current = urlparse(index_url)
    query_params = parse_qs(parsed_current.query)

    current_page = 1
    if "page" in query_params:
        try:
            current_page = int(query_params["page"][0])
        except (ValueError, IndexError):
            current_page = 1

    query_params["page"] = [str(current_page + 1)]

    fallback_url = urlunparse((
        parsed_current.scheme,
        parsed_current.netloc,
        parsed_current.path,
        "",
        urlencode(query_params, doseq=True),
        ""
    ))

    return normalize_url(fallback_url)

## Fungsi mengambil artikel dengan Trafilatura

In [20]:
def scrape_article(url):
    """
    Mengambil isi artikel menggunakan Trafilatura.
    Mengembalikan string isi artikel, atau None jika gagal.
    """

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT
        )

        if response.status_code != 200:
            print(
                f"⚠ Status {response.status_code}: {url}"
            )
            return None

        text = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False,
            include_links=False,
            favor_precision=True
        )

        if not text:
            return None

        text = text.strip()

        # Minimal panjang artikel
        if len(text) < 200:
            return None

        return text

    except requests.RequestException as e:
        print(f"⚠ Request error: {url}")
        print(e)
        return None

    except Exception as e:
        print(f"⚠ Error artikel: {url}")
        print(e)
        return None

## Fungsi utama crawling per kategori

In [21]:
def crawl_category(start_url, label, allowed_domain, target=100):
    results = []
    seen_urls = set()
    seen_index_pages = set()

    index_url = start_url
    page_number = 1

    while len(results) < target and page_number <= MAX_PAGES:

        print("\n" + "="*60)
        print(f"{label.upper()} - HALAMAN INDEX {page_number}")
        print(index_url)
        print("="*60)

        # Hindari halaman index yang sama
        if index_url in seen_index_pages:
            print("⚠ Halaman index sudah pernah dikunjungi.")
            break

        seen_index_pages.add(index_url)

        # Ambil link artikel
        article_links = get_links_from_index(
            index_url,
            allowed_domain=allowed_domain
        )

        print(f"🔎 Ditemukan {len(article_links)} link dari index.")

        if not article_links:
            print("⚠ Tidak ada link artikel pada halaman ini.")
            break

        for article_url in article_links:

            # Berhenti kalau sudah mencapai target
            if len(results) >= target:
                break

            # Hindari duplikasi URL
            if article_url in seen_urls:
                continue

            seen_urls.add(article_url)

            print(f"[{len(results)+1}/{target}] {article_url}")

            article_text = scrape_article(article_url)

            if not article_text:
                print("   ❌ Gagal mengambil artikel")
                continue

            # Simpan
            results.append({
                "id": len(results) + 1,
                "isi_berita": article_text,
                "label": label,
                "link_berita": article_url
            })

            print("   ✅ Berhasil")

            time.sleep(DELAY)

        print(f"\n📊 Total {label}: {len(results)}/{target}")

        # Kalau sudah cukup
        if len(results) >= target:
            break

        # ==========================
        # PAGINATION
        # ==========================

        next_url = get_next_page(index_url, allowed_domain)

        if not next_url:
            print("⚠ Tidak ditemukan pagination.")
            break

        # Jika URL berikutnya sama / sudah pernah dikunjungi
        if next_url in seen_index_pages:
            print("⚠ Pagination kembali ke halaman sebelumnya, berhenti.")
            break

        index_url = next_url
        page_number += 1

    print("\n" + "="*60)
    print(f"SELESAI CRAWLING {label.upper()}")
    print(f"Total berhasil: {len(results)}")
    print("="*60)

    return results

## Menjalankan crawling untuk tiap kategori

In [22]:
sport_index = "https://sport.detik.com/sport-lain/indeks"

finance_index = "https://finance.detik.com/berita-ekonomi-bisnis/indeks"

sport_data = crawl_category(
    sport_index,
    label="sport",
    allowed_domain="sport.detik.com",
    target=TARGET_PER_CATEGORY
)

finance_data = crawl_category(
    finance_index,
    label="finance",
    allowed_domain="finance.detik.com",
    target=TARGET_PER_CATEGORY
)


SPORT - HALAMAN INDEX 1
https://sport.detik.com/sport-lain/indeks
🔎 Ditemukan 20 link dari index.
[1/100] https://sport.detik.com/sport-lain/d-8654641/ambisi-morgan-holindo-jadi-juara-nasional-eshark-rok-cup-2027
   ✅ Berhasil
[2/100] https://sport.detik.com/sport-lain/d-8654586/ihr-2026-perluas-olahraga-pacuan-kuda-indonesia
   ✅ Berhasil
[3/100] https://sport.detik.com/sport-lain/d-8654342/menuju-asian-games-2026-ketum-koi-tekankan-kolaborasi
   ✅ Berhasil
[4/100] https://sport.detik.com/sport-lain/d-8653123/wrt-32-di-lone-star-le-mans-startnya-sudah-bagus-tapi
   ✅ Berhasil
[5/100] https://sport.detik.com/sport-lain/d-8653014/debut-di-rok-cup-indonesia-2026-barra-ghaisan-torehkan-catatan-positif
   ✅ Berhasil
[6/100] https://sport.detik.com/sport-lain/d-8651374/wec-2026-sean-gelael-balapan-di-austin-dini-hari-start-p7
   ✅ Berhasil
[7/100] https://sport.detik.com/sport-lain/d-8650342/wec-2026-wrt-32-meraba-cota-dari-hasil-free-practice
   ✅ Berhasil
[8/100] https://sport.detik.com/

## Menggabungkan hasil crawling

In [23]:
data = sport_data + finance_data

df = pd.DataFrame(data)

# ID otomatis dari 1 sampai total data
if len(df) > 0:
    df["id"] = range(1, len(df) + 1)
    df = df[["id", "isi_berita", "label", "link_berita"]]

print("\nJumlah data:")
print(df["label"].value_counts())

print("\nTotal data:")
print(len(df))


Jumlah data:
label
sport      100
finance    100
Name: count, dtype: int64

Total data:
200


## Menyimpan dataset ke Excel

In [24]:
filename = "dataset_detik_sport_finance.xlsx"

df.to_excel(
    filename,
    index=False
)

print(f"✅ Dataset berhasil disimpan: {filename}")

✅ Dataset berhasil disimpan: dataset_detik_sport_finance.xlsx
